NOME: Herules André

Topico 2 - 1.3

Use Julia/JuMP to solve the problem.

-------------------------------------------------------

IMPORTAR A BIBLIOTECA

In [10]:
using Pkg
Pkg.add("JuMP")

   Resolving package versions...
     Project No packages added to or removed from `C:\Users\hercules.andre\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\hercules.andre\.julia\environments\v1.12\Manifest.toml`


In [9]:
using Pkg
Pkg.add("HiGHS")

   Resolving package versions...
     Project No packages added to or removed from `C:\Users\hercules.andre\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\hercules.andre\.julia\environments\v1.12\Manifest.toml`


In [21]:
using JuMP, HiGHS

PARÂMETROS

In [16]:
# Horzinte de tempo
N = 3

#Dimensões
nx = 3 # estados
nu = 2 # entradas



2

MATRIZ DO SISTEMA

In [19]:
A = [0.2681  -0.00338  -0.00728;
     9.703    0.3279  -25.44;
     0        0        1]

B = [-0.00537  0.1655;
      1.297    97.91;
      0       -6.637]

C = [1.0 0.0 0.0;
     0.0 1.0 0.0;
     0.0 0.0 1.0]  #Matriz Identidade

3×3 Matrix{Float64}:
 1.0  0.0  0.0
 0.0  1.0  0.0
 0.0  0.0  1.0

MODELO DE OTIMIZAÇÃO

In [22]:
model = Model(HiGHS.Optimizer)

A JuMP Model
├ solver: HiGHS
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

VARIÁVEIS DE DECISÃO

In [23]:
# Estados ao longo do tempo

@variable(model, x[1:nx, 0:N])

# Entradas de controle

@variable(model, u[1:nu, 0:N-1])

# Variáveis auxiliares (valor absoluto)

@variable(model, z1[0:N] >= 0)
@variable(model, z3[0:N] >= 0)

1-dimensional DenseAxisArray{VariableRef,1,...} with index sets:
    Dimension 1, 0:3
And data, a 4-element Vector{VariableRef}:
 z3[0]
 z3[1]
 z3[2]
 z3[3]

CONDIÇÃO INICIAL

In [25]:
x0 = [-0.03, 0.0, 0.3]



3-element Vector{Float64}:
 -0.03
  0.0
  0.3

RESTRIÇÕES DOS ESTADOS

In [26]:
x_min = [-0.05, -5.0, -0.5]
x_max = [0.05, 5.0, 0.5]

for k in 0:N
    @constraint(model, x_min .<= x[:,k] .<= x_max)
end

RESTRIÇÕES DAS ENTRADAS

In [27]:
u_min = [-10.0, -0.05]
u_max = [ 10.0,  0.05]

for k in 0:N-1
    @constraint(model, u_min .<= u[:,k] .<= u_max)
end

LINEARIZAÇÃO 

In [28]:
for k in 0:N
    # |c| → x[1]
    @constraint(model, z1[k] >= x[1,k])
    @constraint(model, z1[k] >= -x[1,k])

    # |h| → x[3]
    @constraint(model, z3[k] >= x[3,k])
    @constraint(model, z3[k] >= -x[3,k])
end

FUNÇÃO OBJETIVO

In [29]:
@objective(model, Min, sum(z1[k] + z3[k] for k in 0:N))

z1[0] + z3[0] + z1[1] + z3[1] + z1[2] + z3[2] + z1[3] + z3[3]

RESOLVER O PROBLEMA

In [30]:
optimize!(model)

Running HiGHS 1.14.0 (git hash: 7df0786de3): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline-5 
LP has 37 rows; 26 cols; 53 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 1e+00]
  Bound   [0e+00, 0e+00]
  RHS     [3e-02, 1e+01]
Presolving model
12 rows, 12 cols, 24 nonzeros 0s
12 rows, 12 cols, 24 nonzeros 0s
Presolve reductions: rows 12(-25); columns 12(-14); nonzeros 24(-29) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     3.2999976567e-01 Pr: 6(1.65) 0.0s
          6     3.3000000000e-01 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Simplex   iterations: 6
Objective value     :  3.3000000000e-01
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.01


RESULTADOS

In [32]:
println("Status: ", termination_status(model))

println("\nEstados:")
for k in 0:N
    println("x($k) = ", value.(x[:,k]))
end

println("\nEntradas:")
for k in 0:N-1
    println("u($k) = ", value.(u[:,k]))
end

Status: OPTIMAL

Estados:
x(0) = 1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, Base.OneTo(3)
And data, a 3-element Vector{Float64}:
 -0.03
 -0.0
  0.3
x(1) = 1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, Base.OneTo(3)
And data, a 3-element Vector{Float64}:
 -0.0
 -5.0
 -0.0
x(2) = 1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, Base.OneTo(3)
And data, a 3-element Vector{Float64}:
 -0.0
 -5.0
 -0.0
x(3) = 1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, Base.OneTo(3)
And data, a 3-element Vector{Float64}:
 -0.0
 -5.0
 -0.0

Entradas:
u(0) = 1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, Base.OneTo(2)
And data, a 2-element Vector{Float64}:
 -10.0
  -0.05
u(1) = 1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, Base.OneTo(2)
And data, a 2-element Vector{Float64}:
 -10.0
  -0.05
u(2) = 1-dimensional DenseAxisAr